In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import skew
from scipy.special import boxcox1p
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
train = train.drop('Id', axis=1)
test = test.drop('Id', axis=1)

In [3]:
def splittingg(df):
    df_num = df.select_dtypes(include=['float64', 'int64'])
    df_cat = df.select_dtypes(include=['object'])

    return df_num, df_cat

In [4]:
def feature_engineering(df):
    df_feat = df.copy()

    # Additional useful features
    df_feat['TotalSF'] = (
        df_feat['TotalBsmtSF'].fillna(0)
        + df_feat['1stFlrSF'].fillna(0)
        + df_feat['2ndFlrSF'].fillna(0)
    )

    df_feat['TotalPorchSF'] = (
        df_feat['OpenPorchSF'].fillna(0)
        + df_feat['EnclosedPorch'].fillna(0)
        + df_feat['3SsnPorch'].fillna(0)
        + df_feat['ScreenPorch'].fillna(0)
    )

    df_feat['TotalBath'] = (
        df_feat['FullBath'].fillna(0)
        + 0.5 * df_feat['HalfBath'].fillna(0)
        + df_feat['BsmtFullBath'].fillna(0)
        + 0.5 * df_feat['BsmtHalfBath'].fillna(0)
    )

    # House age
    df_feat['HouseAge'] = (
        df_feat['YrSold'] - df_feat['YearBuilt']
    )

    df_feat['RemodAge'] = (
        df_feat['YrSold'] - df_feat['YearRemodAdd']
    )

    df_feat['IsRemod'] = (df_feat['YearRemodAdd'] != df_feat['YearBuilt']).astype(int)

    df_feat['IsNew'] = (df_feat['YrSold'] == df_feat['YearBuilt']).astype(int)

    # Missing value handling / cleanup
    df_feat['LotFrontage'] = df_feat['LotFrontage'].fillna(
        df_feat['LotFrontage'].median()
    )
    df_feat = df_feat.drop('GarageYrBlt', axis=1)
    df_feat['MasVnrArea'] = df_feat['MasVnrArea'].fillna(0)

    return df_feat

In [5]:


def handle_skewness(df_feat, threshold=0.75, lam=0.15):

    df_feat = df_feat.copy()

    # Log-transform the target
    if 'SalePrice' in df_feat.columns:
        df_feat['SalePrice'] = np.log1p(df_feat['SalePrice'])

    # Compute skewness of numeric features
    numeric_feats = df_feat.select_dtypes(include=['float64', 'int64']).columns
    skewed_features = df_feat[numeric_feats].apply(lambda x: skew(x.dropna()))

    # Select features above the skewness threshold
    skewed = skewed_features[abs(skewed_features) > threshold].index
    skewed = skewed.drop('SalePrice', errors='ignore')

    # Apply Box-Cox transform
    for col in skewed:
        df_feat[col] = boxcox1p(df_feat[col], lam)

    return df_feat

In [6]:
def handle_categorical(df_cat):
    df_cat = df_cat.copy()

    # Columns where NaN likely means 'None' (absence of feature)
    none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']
    for col in none_cols:
        if col in df_cat.columns:
            df_cat[col] = df_cat[col].fillna('0')  # Changed from 'None' to '0'

    # Impute remaining categorical missing values with the mode
    for col in df_cat.columns:
        if df_cat[col].isnull().any():
            df_cat[col] = df_cat[col].fillna(df_cat[col].mode()[0])

    # One-hot encode
    df_cat = pd.get_dummies(df_cat, dtype=int)

    return df_cat

In [7]:
def combined(df_feat, df_cat):
    df_final = pd.concat([df_feat, df_cat], axis=1)
    return df_final

In [8]:
# ---- 1. Run the existing pipeline on train and test ----

train_feat = feature_engineering(train)
test_feat  = feature_engineering(test)

train_num, train_cat = splittingg(train_feat)
test_num,  test_cat  = splittingg(test_feat)

# handle_skewness only touches numeric cols (and log1p's SalePrice if present)
train_num = handle_skewness(train_num)
test_num  = handle_skewness(test_num)   # no SalePrice in test, so it's just boxcox on skewed feats

# ---- 2. Encode categoricals TOGETHER so train/test get identical dummy columns ----

n_train = train_cat.shape[0]
cat_combined = pd.concat([train_cat, test_cat], axis=0, ignore_index=True)
cat_combined_encoded = handle_categorical(cat_combined)

train_cat_enc = cat_combined_encoded.iloc[:n_train, :].reset_index(drop=True)
test_cat_enc  = cat_combined_encoded.iloc[n_train:, :].reset_index(drop=True)

# ---- 3. combining ----

train_final = combined(train_num.reset_index(drop=True), train_cat_enc)
test_final  = combined(test_num.reset_index(drop=True), test_cat_enc)



# ---- 4. Split target out of train_final ----

y = train_final['SalePrice']          
X = train_final.drop('SalePrice', axis=1)

X_test_final = test_final.copy()
if 'SalePrice' in X_test_final.columns:
    X_test_final = X_test_final.drop('SalePrice', axis=1)

# Safety: make sure test has exactly the same columns as train, in the same order
X_test_final = X_test_final.reindex(columns=X.columns, fill_value=0)

# ---- 4.5. Impute any remaining numeric NaNs (test.csv has a few train.csv doesn't) ----

numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns
train_medians = X[numeric_cols].median()

X[numeric_cols] = X[numeric_cols].fillna(train_medians)
X_test_final[numeric_cols] = X_test_final[numeric_cols].fillna(train_medians)

In [9]:


def scale_features(X_train, X_test=None):
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
    )
    if X_test is not None:
        X_test_scaled = pd.DataFrame(
            scaler.transform(X_test), columns=X_test.columns, index=X_test.index
        )
        return X_train_scaled, X_test_scaled, scaler
    return X_train_scaled, scaler


def train_lasso(X_train_scaled, y_train, random_state=42):
    model = Lasso(alpha=0.005428, random_state=random_state, max_iter=10000)
    model.fit(X_train_scaled, y_train)
    return model

def predict_lasso(model, X_scaled, inverse_log=True):
    preds = model.predict(X_scaled)
    if inverse_log:
        preds = np.expm1(preds)
    return preds

In [10]:
# ==== Deployment export: fit on ALL of train.csv, save model + preprocessor with joblib ====
import joblib

NONE_COLS = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']

train_raw = pd.read_csv('../data/train.csv').drop('Id', axis=1)
raw_feature_cols = [c for c in train_raw.columns if c != 'SalePrice']

lot_frontage_median = train_raw['LotFrontage'].median()

raw_num_cols = list(train_raw.drop('SalePrice', axis=1).select_dtypes(include=['float64', 'int64']).columns)
raw_cat_cols = list(train_raw.select_dtypes(include=['object']).columns)
raw_numeric_defaults = train_raw[raw_num_cols].median().to_dict()
raw_categorical_defaults = {
    c: (train_raw[c].mode()[0] if not train_raw[c].mode().empty else '')
    for c in raw_cat_cols
}
for c in NONE_COLS:
    if c in raw_categorical_defaults and train_raw[c].isnull().any():
        raw_categorical_defaults[c] = '0'
categorical_options = {c: sorted(train_raw[c].dropna().unique().tolist()) for c in raw_cat_cols}

train_feat = feature_engineering(train_raw)
train_num, train_cat = splittingg(train_feat)

train_num_log = train_num.copy()
train_num_log['SalePrice'] = np.log1p(train_num_log['SalePrice'])
numeric_feats = train_num_log.select_dtypes(include=['float64', 'int64']).columns
skewed_series = train_num_log[numeric_feats].apply(lambda x: skew(x.dropna()))
skewed_cols = list(skewed_series[abs(skewed_series) > 0.75].index.drop('SalePrice', errors='ignore'))

train_num_sk = handle_skewness(train_num)

cat_modes = {}
train_cat_check = train_cat.copy()
for col in train_cat_check.columns:
    if col in NONE_COLS:
        train_cat_check[col] = train_cat_check[col].fillna('0')
    if train_cat_check[col].isnull().any():
        cat_modes[col] = train_cat_check[col].mode()[0]

train_cat_enc = handle_categorical(train_cat)

train_final = combined(train_num_sk, train_cat_enc)
y = train_final['SalePrice']
X = train_final.drop('SalePrice', axis=1)

numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns
numeric_medians = X[numeric_cols].median()
X[numeric_cols] = X[numeric_cols].fillna(numeric_medians)

final_columns = list(X.columns)

X_full_scaled, scaler_full = scale_features(X)
final_model = train_lasso(X_full_scaled, y)

preprocessor = {
    'raw_feature_cols': raw_feature_cols,
    'raw_numeric_cols': raw_num_cols,
    'raw_categorical_cols': raw_cat_cols,
    'raw_numeric_defaults': raw_numeric_defaults,
    'raw_categorical_defaults': raw_categorical_defaults,
    'categorical_options': categorical_options,
    'lot_frontage_median': lot_frontage_median,
    'none_cols': NONE_COLS,
    'cat_modes': cat_modes,
    'skewed_cols': skewed_cols,
    'boxcox_lambda': 0.15,
    'numeric_medians': numeric_medians.to_dict(),
    'final_columns': final_columns,
    'scaler': scaler_full,
}

joblib.dump(final_model, 'model.joblib')
joblib.dump(preprocessor, 'preprocessor.joblib')
print("Saved model.joblib and preprocessor.joblib —", len(final_columns), "features")

Saved model.joblib and preprocessor.joblib — 299 features


In [ ]:
# ---- 5. Optional: hold out a validation split from train to sanity-check before predicting on test ----


X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

X_tr_scaled, X_val_scaled, scaler = scale_features(X_tr, X_val)
lasso_model = train_lasso(X_tr_scaled, y_tr)

val_preds_log = predict_lasso(lasso_model, X_val_scaled, inverse_log=False)


# ---- 6. Refit on ALL of train, then predict on the actual test set ----
X_full_scaled, X_test_scaled, scaler_full = scale_features(X, X_test_final)
final_model = train_lasso(X_full_scaled, y)

test_preds = predict_lasso(final_model, X_test_scaled, inverse_log=True)  # back to $ scale

# ---- 7. Build submission ----
submission = pd.DataFrame({
    'Id': pd.read_csv('../data/test.csv')['Id'],   # re-read 
    'SalePrice': test_preds
})
submission.to_csv('../data/submission.csv', index=False)

# FINSHED THE PROJECT